#### Plotting simple profiles of the ocean currents at a given dFAD time. 
1. Simple profile plots
    - Plot of the entire domain currents
    - plot of vertical profile of speed 
2. Making Annimation of dFAD traveling over current


In [1]:
import numpy as np 
import pandas as pd 
import xarray as xr 
import matplotlib.pyplot as plt 
import geopandas as gpd
from functions.funcs import *
import matplotlib.animation as animation
import imageio_ffmpeg
from matplotlib import rcParams
rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
%matplotlib notebook

In [2]:
import imageio_ffmpeg
from matplotlib import rcParams

rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

In [3]:
ds = xr.open_dataset(r"Data\cmems(3).nc")
data = gpd.read_parquet(r"Data\Mapped_4hr_period.parquet")
#data2 = gpd.read_parquet(r"Data\Mapped_4hr_period2.parquet")

In [4]:
data = Add_x_y_speed_collums(data)

In [5]:
vo  = ds['vo']
uo = ds['uo']

In [6]:
## plotting functions - First grad a test time 
data = add_time_collumns(data)
test_time = data.at[1,"timelist"][0]
print(type(test_time))

<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [7]:
def Plotting_xy_currents(uo: xr.Dataset, vo : xr.Dataset, time: pd.Timestamp, depth: int, ax : plt.Axes):
    """Plot to create a profile of whole domains currents at specified depth"""
    ufield = uo.sel( depth = depth, time = time,method = "nearest")
    vfield = vo.sel( depth = depth, time = time,method = "nearest")
    latitude = uo["latitude"].to_numpy()
    longitude = uo["longitude"].to_numpy()
    latlocs, lonlocs = np.meshgrid( longitude, latitude)
    plot = ax.quiver(latlocs,lonlocs ,ufield,  vfield,headaxislength = 3, headlength = 3)
    ax.set_title("Model Currents")
    return plot
    


In [8]:
def Plotting_vertical_profile(uo: xr.Dataset, vo : xr.Dataset, lat : float, lon :float , time: pd.Timestamp,  ax : plt.Axes): 
    """Plots a vertial porfile of """
    ufield = uo.sel( longitude = lon, latitude = lat, time = time,method = "nearest")
    vfield = vo.sel( longitude = lon, latitude = lat, time = time,method = "nearest")
    yprofile = ax.plot(vfield,-vfield["depth"], label = "y speed", color = "g")
    xprofile = ax.plot(ufield,-ufield["depth"], label = "x speed", color = "b")
    ax.axvline(x = 0, alpha = 0.18, color = "k") #ymax = -max(vfield["depth"]), ymin = min(vfield["depth"]),
    ax.grid(alpha = 0.1)
    ax.set_ylabel("Depth")
    ax.set_xlabel("Speed")
    #ax.legend(loc = 'upper right')
    return xprofile, yprofile


In [9]:
def Add_lat_lon_collumns(data = pd.DataFrame):
    lats = []
    lons = []
    for i in range(len(data)):
        lat = []
        lon = []
        line = data.at[i,"geometry"]
        x,y = line.xy
        lats.append(y)
        lons.append(x)
    data["lat"] = lats
    data["lon"] = lons
    return data

In [10]:
def OneTrajectory(ax, ds,index, window:int = None , itime:int =None):
    line = ds.at[index,'geometry']
    x,y = line.xy
    if window !=None: ## adds padding to end of the array for sliding window
        x= np.array(x)
        y=np.array(y)
        nans = np.full(window,np.nan)
        x = np.concatenate((x,nans))
        nans = np.full(window,np.nan)
        y = np.concatenate((y,nans))
        x= x[itime:itime+window]
        y = y[itime:itime+window]
    dfadline = ax.plot(x,y)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("latitude")
    return dfadline

In [11]:
data = Add_lat_lon_collumns(data)
test_lat = data.at[1,"lat"][0]
test_lon = data.at[1,"lon"][0]
fig, ax = plt.subplots()
ax = Plotting_vertical_profile(uo,vo,test_lat,test_lon,time = test_time, ax= ax)

<IPython.core.display.Javascript object>

In [12]:
fig, ax = plt.subplots()
plot= Plotting_xy_currents(uo, vo, test_time, 15, ax)
ax = OneTrajectory(ax, data, 1,window = 5, itime= 0)

<IPython.core.display.Javascript object>

In [23]:
data.query("Diff_days > 60")


,level_0,index,Shape_Leng,Set_3,BuoyName,Name_ID,MinOfTimes,MaxOfTimes,MinOfDate,MaxOfDate,...,y_km,xy_km,timelist,mapped_v,mapped_u,x_speed,y_speed,xy_speed,lat,lon
50,50,1120,7.322212,1268.0,SLX+391231,391231.0,45083.316667,45143.817361,2023-06-06,2023-08-05,...,"[-0.22071584076147818, -1.9789179312651557, -0...","[0.6974536047564752, 4.177765665390995, 2.8036...","[2023-06-06 07:36:00.002879728, 2023-06-06 11:...","[0.05990370026367006, 0.08031826672149955, 0.1...","[-0.01094197809026664, -0.01575227145074546, -...","[0.046198267129812516, 0.256924480558054, 0.19...","[-0.0154119379166551, -0.13818201808076636, -0...","[0.04870113363485489, 0.2917210873637345, 0.19...","[6.724050000000034, 6.7219800000000305, 6.7034...","[-163.51836999999998, -163.51241999999996, -16..."
117,117,1463,7.403287,1619.0,SLX+404595,404595.0,45080.259722,45142.259722,2023-06-03,2023-08-04,...,"[0.20587801281883608, -0.18880529450398068, 0....","[0.6211098144359635, 1.568135826396551, 0.8828...","[2023-06-03 06:13:59.998079719, 2023-06-03 10:...","[0.20749392193493296, 0.2207597941730775, 0.23...","[-0.0835706966216388, -0.06261979473643062, -0...","[0.04047546809013113, 0.10752496266770455, 0.0...","[0.014220218179317757, -0.013040986963573935, ...","[0.04290073016377437, 0.10831284643196012, 0.0...","[6.5834200000000465, 6.585350000000062, 6.5835...","[-163.60309999999998, -163.59782999999996, -16..."
132,132,1525,9.748172,1683.0,SLX+408964,408964.0,45068.220833,45129.554167,2023-05-22,2023-07-22,...,"[0.16168927013421383, 0.6457269173239233, -0.1...","[0.5822133926859303, 2.3317811031106044, 3.232...","[2023-05-22 05:17:59.997120145, 2023-05-22 09:...","[0.10505920446612471, 0.0963928381857349, 0.08...","[0.019248207105523305, 0.020924029345667207, 0...","[-0.03905422056033798, -0.1564498099985017, -0...","[0.011290059157362704, 0.04508830480858177, -0...","[0.040653431363607134, 0.16281814231882386, 0....","[7.597080000000062, 7.5986200000000395, 7.6047...","[-160.77364999999998, -160.77867999999998, -16..."


#### want to make an animation with that changes through time of the currents 

In [24]:
## first get a the time I want to animate  
dfadidex = 50
times = data.at[dfadidex,"timelist"]
artists = []
fig, ax = plt.subplots()

for n,time in enumerate(times): ## giving n doesnt quite work because if times is not the start time. 
    current = Plotting_xy_currents(uo, vo, time, 15, ax)
    dfad = OneTrajectory(ax, data,dfadidex,window = 12,itime=n,)
    ann = ax.annotate(f"{times[n]:%Y :%m :%d :%H}",(0.15, 1.03), xycoords="axes fraction", ha="center")
    artists.append([current,dfad[0],ann])


print(artists)

ani = animation.ArtistAnimation(fig =fig, artists = artists, blit= False)
ani.save(rf"..\Figures\Animations\Animation_{dfadidex}.mp4", writer = "ffmpeg")

<IPython.core.display.Javascript object>

[[<matplotlib.quiver.Quiver object at 0x0000027E92A73890>, <matplotlib.lines.Line2D object at 0x0000027E92A739D0>, Text(0.15, 1.03, '2023 :06 :06 :07')], [<matplotlib.quiver.Quiver object at 0x0000027E92A73C50>, <matplotlib.lines.Line2D object at 0x0000027E92A73D90>, Text(0.15, 1.03, '2023 :06 :06 :11')], [<matplotlib.quiver.Quiver object at 0x0000027E92B14050>, <matplotlib.lines.Line2D object at 0x0000027E92B14190>, Text(0.15, 1.03, '2023 :06 :06 :15')], [<matplotlib.quiver.Quiver object at 0x0000027E92B14410>, <matplotlib.lines.Line2D object at 0x0000027E92B14550>, Text(0.15, 1.03, '2023 :06 :06 :19')], [<matplotlib.quiver.Quiver object at 0x0000027E92B147D0>, <matplotlib.lines.Line2D object at 0x0000027E92B14910>, Text(0.15, 1.03, '2023 :06 :06 :23')], [<matplotlib.quiver.Quiver object at 0x0000027E92B14B90>, <matplotlib.lines.Line2D object at 0x0000027E92B14CD0>, Text(0.15, 1.03, '2023 :06 :07 :03')], [<matplotlib.quiver.Quiver object at 0x0000027E92B14F50>, <matplotlib.lines.Line2

#### adding profile plot to the side also 

In [14]:
from matplotlib.gridspec import GridSpec

In [15]:
dfadidex = 109
times = data.at[dfadidex,"timelist"]
lats = data.at[dfadidex, "lat"]
lons = data.at[dfadidex, "lon"]
artists = []

fig = plt.figure()
gs = GridSpec(3,3)
ax1 = fig.add_subplot(gs[:,:-1])
ax2 = fig.add_subplot(gs[:,-1])

for n,time in enumerate(times): ## giving n doesnt quite work because if times is not the start time. 
    current = Plotting_xy_currents(uo, vo, time, 15, ax1)
    dfad = OneTrajectory(ax1, data,dfadidex,window = 12,itime=n,)
    ann = ax1.annotate(f"{times[n]:%Y :%m :%d :%H}",(0.10, 1.03), xycoords="axes fraction", ha="center")
    xprofile, yprofile = Plotting_vertical_profile(uo,vo,lat = lats[n], lon= lons[n], time = times[n], ax=  ax2)
    artists.append([current,dfad[0],ann, xprofile[0], yprofile[0]])


print(artists)

ani = animation.ArtistAnimation(fig =fig, artists = artists, blit= False)
ani.save(rf"..\Figures\Animations\Depths_{dfadidex}.mp4", writer = "ffmpeg")

<IPython.core.display.Javascript object>

[[<matplotlib.quiver.Quiver object at 0x0000027E8B305810>, <matplotlib.lines.Line2D object at 0x0000027E8B305950>, Text(0.1, 1.03, '2023 :07 :05 :15'), <matplotlib.lines.Line2D object at 0x0000027E8B305D10>, <matplotlib.lines.Line2D object at 0x0000027E8B305BD0>], [<matplotlib.quiver.Quiver object at 0x0000027E8B305F90>, <matplotlib.lines.Line2D object at 0x0000027E8B3060D0>, Text(0.1, 1.03, '2023 :07 :05 :19'), <matplotlib.lines.Line2D object at 0x0000027E8B306490>, <matplotlib.lines.Line2D object at 0x0000027E8B306350>], [<matplotlib.quiver.Quiver object at 0x0000027E8B306710>, <matplotlib.lines.Line2D object at 0x0000027E8B306850>, Text(0.1, 1.03, '2023 :07 :05 :22'), <matplotlib.lines.Line2D object at 0x0000027E8B306C10>, <matplotlib.lines.Line2D object at 0x0000027E8B306AD0>], [<matplotlib.quiver.Quiver object at 0x0000027E8B306E90>, <matplotlib.lines.Line2D object at 0x0000027E8B306FD0>, Text(0.1, 1.03, '2023 :07 :06 :02'), <matplotlib.lines.Line2D object at 0x0000027E8B307390>, 

In [16]:
first = data["x_km"][40]
first

array([-0.91068645, -2.52746068, -3.75616462, -4.45558071, -2.41848965,
       -4.90703211, -5.33735648, -2.41292991, -4.19760848, -4.0964211 ,
       -2.71982791, -4.10198084, -4.27544493, -2.90329953, -4.04415948,
       -4.08641355, -4.91259186, -6.07569079, -3.86513565, -3.98077837,
       -4.68909006, -3.82176963, -4.36551282, -4.04193558, -0.613796  ,
       -3.85846395, -3.638298  , -2.54413992, -6.00119019, -3.65275334,
       -0.18903138, -4.85921829, -4.97708492, -3.39144526, -4.41666249,
       -4.66129132, -2.36289219, -2.40181042, -4.08196576, -3.80509039,
       -3.32472831, -4.05527897, -4.5545442 , -5.23060935, -3.59493198,
       -4.8503227 , -3.88737464, -3.89960608, -4.1664739 , -4.75135922,
       -3.87847904, -1.88364206, -2.39402677, -4.6201492 , -2.87216496,
       -3.51820748, -4.93816669, -3.09344286, -3.29136983, -4.36773672,
       -4.11310034, -4.91036796, -5.13386976, -3.93963625, -3.38588552,
       -4.51117817, -5.50748472, -4.48115554, -4.01636075, -3.31